In [94]:
# !pip install scikeras

# # Uninstall the current scikit-learn version
# !pip uninstall scikit-learn -y

# # Install a compatible version of scikit-learn (e.g., 1.4.2)
# !pip install scikit-learn==1.4.2

In [95]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


In [96]:
# Read training data
train_df = pd.read_csv('train.csv')

In [97]:
# Get decision tree feature importance
drop_columns = ['color', 'car_name', 'map_code', 'assists', 'mvp', 'demos_inflicted', 'demos_taken']
match_df = train_df.drop(columns=drop_columns).groupby(['match_id', 'rank']).mean().reset_index()

X = match_df.drop(columns=['match_id', 'rank'])
y = match_df['rank']

X_tree_train, X_tree_test, y_tree_train, y_tree_test = train_test_split(
    X, y,
    test_size = 0.25,
    random_state = 42,
    stratify=y
)

ct_tree = ColumnTransformer(
    transformers=[
        ('imputer', SimpleImputer(), ['possession_time', 'time_in_side'])
    ],
    remainder='passthrough'
)

pipe_tree = Pipeline(
    steps = [
        ('transformer', ct_tree),
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(max_depth=5))
    ]
).fit(X_tree_train, y_tree_train)

y_pred_tree_train = pipe_tree.predict(X_tree_train)
y_pred_tree_test = pipe_tree.predict(X_tree_test)

print(classification_report(
    y_true=y_tree_train,
    y_pred=y_pred_tree_train
))
print(classification_report(
    y_true=y_tree_test,
    y_pred=y_pred_tree_test
))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

      bronze       0.00      0.00      0.00       553
    champion       0.66      0.69      0.68      4408
     diamond       0.48      0.41      0.44      5187
        gold       0.49      0.54      0.51      4689
    platinum       0.45      0.53      0.49      5623
      silver       0.52      0.41      0.46      2130

    accuracy                           0.51     22590
   macro avg       0.43      0.43      0.43     22590
weighted avg       0.50      0.51      0.50     22590

              precision    recall  f1-score   support

      bronze       0.00      0.00      0.00       184
    champion       0.64      0.66      0.65      1470
     diamond       0.45      0.40      0.42      1729
        gold       0.47      0.51      0.49      1563
    platinum       0.43      0.51      0.46      1875
      silver       0.51      0.38      0.44       710

    accuracy                           0.49      7531
   macro avg       0.42

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [98]:
features_df = pd.DataFrame(
    data={
        'feature': pipe_tree[:-1].get_feature_names_out(),
        'importance': pipe_tree.named_steps['model'].feature_importances_
    }
)
important_features_df = features_df.sort_values('importance')
important_features_df

,feature,importance
20,remainder__count_stolen_big,0.000000
68,remainder__goals_against_while_last_defender,0.000000
6,remainder__goals_against,0.000022
2,remainder__duration,0.000026
67,remainder__time_most_forward,0.000047
...,...,...
53,remainder__percent_ground,0.078676
40,remainder__time_supersonic_speed,0.082876
11,remainder__bcpm,0.086765
54,remainder__percent_low_air,0.088053


In [99]:
le = LabelEncoder()
y = le.fit_transform(y)
le.classes_

array(['bronze', 'champion', 'diamond', 'gold', 'platinum', 'silver'],
      dtype=object)

In [100]:
# Split train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [101]:
# Initialize column transformer
ct = ColumnTransformer(
    transformers=[
        ('imputer', SimpleImputer(), X.columns)
    ],
    remainder='passthrough'
)

In [102]:
es = EarlyStopping(monitor='val_loss', patience=2)

In [103]:
# Function to create the Keras model for SciKeras
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(len(X.columns),)))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(6, activation='softmax')) # Need output node per rank
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    epochs=100,
    validation_split=0.1,
    callbacks=[es]
)

In [104]:
# Create and fit model pipeline
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('scaler', StandardScaler()),
        ('model', model)
    ]
).fit(X_train, y_train)

Epoch 1/100
678/678 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5211 - loss: 1.0796 - val_accuracy: 0.5573 - val_loss: 1.0021
Epoch 2/100
678/678 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5561 - loss: 0.9915 - val_accuracy: 0.5539 - val_loss: 0.9968
Epoch 3/100
678/678 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5638 - loss: 0.9751 - val_accuracy: 0.5469 - val_loss: 0.9927
Epoch 4/100
678/678 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5706 - loss: 0.9621 - val_accuracy: 0.5519 - val_loss: 0.9772
Epoch 5/100
678/678 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5718 - loss: 0.9514 - val_accuracy: 0.5506 - val_loss: 0.9916
Epoch 6/100
678/678 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5777 - loss: 0.9443 - val_accuracy: 0.5573 - val_loss: 0.9807


In [105]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

753/753 ━━━━━━━━━━━━━━━━━━━━ 1s 916us/step
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [106]:
confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
)

array([[ 281,    0,    6,   58,   21,  224],
       [   0, 3244, 1341,    6,  111,    0],
       [   2,  948, 3252,  101, 1228,    2],
       [  40,    8,  162, 2881, 1151,  759],
       [  11,   82, 1429, 1302, 3102,   72],
       [ 145,    0,   13,  616,   66, 1432]])

In [107]:
confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)

array([[ 55,   0,   4,  16,   8,  64],
       [  0, 743, 397,   0,  36,   0],
       [  0, 260, 771,  41, 311,   0],
       [ 11,   0,  60, 672, 302, 206],
       [  4,  17, 394, 341, 718,  26],
       [ 38,   0,   2, 174,  26, 328]])

In [108]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))
print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

           0       0.59      0.48      0.53       590
           1       0.76      0.69      0.72      4702
           2       0.52      0.59      0.55      5533
           3       0.58      0.58      0.58      5001
           4       0.55      0.52      0.53      5998
           5       0.58      0.63      0.60      2272

    accuracy                           0.59     24096
   macro avg       0.60      0.58      0.59     24096
weighted avg       0.59      0.59      0.59     24096

              precision    recall  f1-score   support

           0       0.51      0.37      0.43       147
           1       0.73      0.63      0.68      1176
           2       0.47      0.56      0.51      1383
           3       0.54      0.54      0.54      1251
           4       0.51      0.48      0.50      1500
           5       0.53      0.58      0.55       568

    accuracy                           0.55      6025
   macro avg       0.55

In [109]:
test_df = pd.read_csv('test.csv')

In [110]:
match_test_df = test_df.drop(columns=drop_columns).groupby(['match_id']).mean().reset_index()

In [111]:
y_pred = pipe.predict(match_test_df[X.columns])

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [112]:
le.classes_

array(['bronze', 'champion', 'diamond', 'gold', 'platinum', 'silver'],
      dtype=object)

In [113]:
converter = { 0: 1, 5: 2, 3: 3, 4: 4, 2: 5, 1: 6 }

y_pred = pd.Series(y_pred).map(converter)

In [114]:
submission = pd.concat([match_test_df['match_id'], y_pred], axis = 1).rename(columns = {0: 'rank'})
submission.head()

,match_id,rank
0,30121,5
1,30122,2
2,30123,5
3,30124,5
4,30125,5


In [117]:
# submission.to_csv('keras_mean_important_features_3-128_relu_submission_4.csv', index=False)

In [116]:
submission.shape

(2500, 2)